# ✦ StoryForge on Kaggle (backup)

**Before you start (right side panel → Settings):** Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified Kaggle account).

1. Run **1️⃣ Setup**.  2. Run **2️⃣ Start** and open the **public link** it prints (Kaggle has no built-in link).
3. Your files are in `/kaggle/working/StoryForge`. **Download your videos before you stop the session**: Kaggle deletes them unless you use *Save Version*.

Optional keys: **Add-ons → Secrets** (`GEMINI_API_KEY`, `GROQ_API_KEY`, `GITHUB_TOKEN` if your repo is private).

In [ ]:
# 1️⃣ Setup (run once per session)

import os, subprocess, sys
REPO = "bokaling/Projectgeming"
APP = "/kaggle/working/storyforge-app"

def _git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True).returncode == 0

def _url(token=None):
    return f"https://{token + '@' if token else ''}github.com/{REPO}.git"

def get_code(token=None):
    if os.path.isdir(APP + "/.git"):  # already downloaded: update it
        branch = subprocess.run(["git", "-C", APP, "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
        return _git("-C", APP, "pull", "-q", "--ff-only", _url(token), branch if branch != "HEAD" else "main") or True
    if not _git("clone", "-q", "--depth", "1", _url(token), APP):
        return False
    if not os.path.exists(APP + "/storyforge/__init__.py"):  # code not merged into main yet: use the build branch
        _git("-C", APP, "fetch", "-q", "--depth", "1", _url(token), "claude/nice-noether-0u7lld:claude/nice-noether-0u7lld")
        _git("-C", APP, "checkout", "-q", "claude/nice-noether-0u7lld")
    _git("-C", APP, "remote", "set-url", "origin", _url())  # never keep a token on disk
    return os.path.exists(APP + "/storyforge/__init__.py")

print("📥 Downloading StoryForge…")
ok = get_code()
if not ok:
    try:
        from kaggle_secrets import UserSecretsClient
        ok = get_code(UserSecretsClient().get_secret("GITHUB_TOKEN"))
    except Exception:
        ok = False
if not ok:
    raise SystemExit("❌ Couldn't download StoryForge. Make your GitHub repository public, or add a GITHUB_TOKEN secret "
                     "(Add-ons → Secrets), and check that Internet is ON in Settings.")
sys.path.insert(0, APP)
from storyforge import install
install.setup()
print("👉 Now run cell 2️⃣")

In [ ]:
# 2️⃣ Start StoryForge: open the public link printed below (don't share it)
import sys
sys.path.insert(0, "/kaggle/working/storyforge-app")
import storyforge
storyforge.launch(public_link=True)

In [ ]:
# 🧰 Backup mode: simple forms (only if the public link doesn't work). Change the values, then run.
ACTION = "⚡ Quick test"   # "⚡ Quick test", "1. Write script", "2. Make pictures", "3. Make video", "🔄 Regenerate scenes"
SERIES, EPISODE, IDEA = "My Series", 1, ""
LANGUAGES, FORMATS = "English", "9:16 + 16:9"      # "English" / "Farsi" / "English + Farsi"
SCENES, HERO_SCENES, QUALITY = 10, 2, "1080p"
ENGLISH_VOICE, FARSI_VOICE, REGENERATE = "kokoro:am_fenrir", "edge:fa-IR-FaridNeural", ""
import sys
sys.path.insert(0, "/kaggle/working/storyforge-app")
from storyforge import forms
forms.run(ACTION, SERIES, EPISODE, IDEA, LANGUAGES, FORMATS, SCENES, HERO_SCENES, QUALITY, ENGLISH_VOICE, FARSI_VOICE, REGENERATE)